# Docs Watcher Agent: Long-Running Loops with State

In this notebook, we build a stateful agent that monitors documentation pages for changes, classifies their severity using both heuristic rules and LLM-based analysis, and maintains persistent memory across ticks. This pattern teaches the fundamentals of long-running agent loops with state persistence.

## Architecture Overview

This notebook teaches **long-running agent patterns** for monitoring and alerting. Unlike request-response agents that answer a single question and stop, a docs watcher agent runs continuously (or on a schedule), maintaining state between executions.

We demonstrate two approaches:

- **Approach 1: Stateful Worker (Pull-Based)** -- Polls pages on a schedule, diffs content against the last known snapshot, and stores state in a database. This is the most common pattern for monitoring external pages you don't control.

- **Approach 2: Push-Based Mechanism** -- Simulates a webhook/event-driven processing model. Instead of polling, the agent receives notifications when content changes. This is more efficient but requires infrastructure support.

### Key Concepts

- **Idempotency**: Running the same tick twice produces the same result (no duplicate alerts)
- **Persistent State**: The agent remembers what it has already seen across runs
- **Diffing**: Compute exactly what changed between two versions of a page
- **Severity Classification**: Combine heuristic rules with optional LLM classification
- **Backoff**: Automatically skip sources that repeatedly fail

### Poll-Based Loop Architecture

```
[Schedule/Tick] -> [Fetch Page] -> [Hash Content] -> [Compare to Last Hash]
                                                            |
                                                +-----------+-----------+
                                          [No Change]           [Change Detected]
                                                |                       |
                                          [Log & Skip]        [Compute Diff]
                                                                        |
                                                                [Classify Severity]
                                                                        |
                                                                [Store Event]
                                                                        |
                                                                [Notify/Alert]
```

Each "tick" of the agent follows this flow. The agent is designed so that any tick can be re-run safely without producing duplicate events.

## Key Concepts in Detail

### Idempotency
The same tick run twice produces the same result -- no duplicate alerts. We achieve this by tracking the last hash we alerted on per source. If the current content hash matches the last alerted hash, we skip alerting even though a change was previously detected.

### Content Hashing
We use SHA-256 to detect changes efficiently. Instead of comparing entire page contents character-by-character, we hash the content and compare short fixed-length strings. This is fast and reliable for change detection.

### Diffing
When a change IS detected, we compute a unified diff showing exactly what lines were added, removed, or modified. This gives the severity classifier (and the human operator) precise context about what changed.

### Severity Classification
We use a two-tier approach:
1. **Heuristic rules**: Fast keyword matching (e.g., "deprecated" = HIGH, "new feature" = MEDIUM)
2. **LLM classification**: For more nuanced understanding, we can optionally send the diff to GPT-4o for classification

### Operational Memory
For each source, we track: last content hash, failure count, last fetch time, and last alerted hash. This is stored in a key-value table in SQLite.

### Backoff
If a source fails 3 or more times consecutively, the agent skips it on future ticks. This prevents hammering a broken endpoint and wasting resources.

In [ ]:
# DEPENDENCY: pip install -q openai pydantic
# (packages should be pre-installed in venv before running this script)

In [ ]:
import os
import json
import hashlib
import sqlite3
import difflib
from datetime import datetime
from getpass import getpass
from pydantic import BaseModel, Field
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY")

client = OpenAI()
MODEL = "gpt-4o"


class SeverityClassification(BaseModel):
    """Structured output for change severity classification."""
    severity: str = Field(description="Severity level: HIGH, MEDIUM, or LOW")
    reason: str = Field(description="Brief explanation for the severity rating")

## Create the State Database

The agent needs persistent storage to remember what it has seen. We use SQLite with four tables:

- **sources**: The documentation pages we monitor (name, URL, active flag)
- **snapshots**: Every fetched version of a page (content hash, full text, timestamp)
- **events**: Detected changes with severity classification and diff
- **kv**: A key-value store for per-source operational metadata (last alerted hash, failure count, etc.)

We use an in-memory database for this demo, but in production you would use a file-based SQLite database or a more robust store like PostgreSQL or Redis.

In [ ]:
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row

conn.executescript("""
CREATE TABLE sources (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    url TEXT NOT NULL,
    active BOOLEAN DEFAULT 1
);

CREATE TABLE snapshots (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    source_id INTEGER,
    fetched_at TEXT,
    content_hash TEXT,
    content_text TEXT,
    FOREIGN KEY (source_id) REFERENCES sources(id)
);

CREATE TABLE events (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    source_id INTEGER,
    created_at TEXT,
    severity TEXT,
    summary TEXT,
    diff_text TEXT,
    FOREIGN KEY (source_id) REFERENCES sources(id)
);

CREATE TABLE kv (
    source_id INTEGER,
    key TEXT,
    value TEXT,
    PRIMARY KEY (source_id, key),
    FOREIGN KEY (source_id) REFERENCES sources(id)
);
""")

print("Database schema created successfully.")
print("Tables: sources, snapshots, events, kv")

In [ ]:
# Register documentation sources to monitor
sources = [
    (1, "OpenAI API Changelog", "https://platform.openai.com/docs/changelog"),
    (2, "Anthropic Release Notes", "https://docs.anthropic.com/en/docs/about-claude/models"),
    (3, "Local Test Source", "local://test"),  # For deterministic demo
]
conn.executemany("INSERT INTO sources (id, name, url) VALUES (?, ?, ?)", sources)
conn.commit()

print("Registered sources:")
for row in conn.execute("SELECT * FROM sources").fetchall():
    print(f"  [{row['id']}] {row['name']} -> {row['url']}")

## Simulated Content

Since we cannot guarantee that real documentation pages will change during the execution of this notebook, we use simulated content with two versions per source. This gives us deterministic control over when "changes" occur, making the demo reproducible.

In production, the `fetch_content` function would make real HTTP requests to the monitored URLs.

In [ ]:
# Simulated page content for deterministic demos
simulated_content = {
    1: {  # OpenAI
        "v1": """# OpenAI API Changelog\n\n## January 2026\n- Released GPT-5.2 with improved reasoning\n- Added new function calling features\n- Updated rate limits for Tier 3 users""",
        "v2": """# OpenAI API Changelog\n\n## February 2026\n- BREAKING: Deprecated gpt-4-turbo model\n- New pricing for GPT-5.2: $2/1M input tokens\n- Released o4-mini reasoning model\n\n## January 2026\n- Released GPT-5.2 with improved reasoning\n- Added new function calling features\n- Updated rate limits for Tier 3 users"""
    },
    2: {  # Anthropic
        "v1": """# Claude Models\n\n## Claude 4.5 Sonnet\n- 200K context window\n- Best for coding and analysis\n- $3/$15 per 1M tokens""",
        "v2": """# Claude Models\n\n## Claude 4.6 Opus\n- 200K context window\n- Most capable model\n- $15/$75 per 1M tokens\n\n## Claude 4.5 Sonnet\n- 200K context window\n- Best for coding and analysis\n- $3/$15 per 1M tokens"""
    },
    3: {  # Test source
        "v1": "Test content version 1",
        "v2": "Test content version 2 - with changes"
    }
}

# Track which version each source is on
current_versions = {1: "v1", 2: "v1", 3: "v1"}


def fetch_content(source_id):
    """Simulate fetching page content.
    
    In production, this would use requests.get(url) to fetch real pages.
    For this demo, we return simulated content based on the current version.
    """
    version = current_versions.get(source_id, "v1")
    return simulated_content.get(source_id, {}).get(version, "")


print("Simulated content loaded.")
print(f"All sources start at version v1.")
print(f"Source 1 (OpenAI) v1 preview: {simulated_content[1]['v1'][:60]}...")
print(f"Source 2 (Anthropic) v1 preview: {simulated_content[2]['v1'][:60]}...")

## Helper Functions

These utility functions handle the core operations: hashing content, retrieving the last snapshot, computing diffs, and managing the key-value store for per-source metadata.

In [ ]:
def compute_hash(content):
    """Compute SHA-256 hash of content for efficient change detection."""
    return hashlib.sha256(content.encode()).hexdigest()


def get_last_snapshot(source_id):
    """Retrieve the most recent snapshot for a source."""
    row = conn.execute(
        "SELECT * FROM snapshots WHERE source_id = ? ORDER BY fetched_at DESC LIMIT 1",
        (source_id,)
    ).fetchone()
    return dict(row) if row else None


def compute_diff(old_text, new_text):
    """Compute a unified diff between two text versions."""
    diff = difflib.unified_diff(
        old_text.splitlines(), new_text.splitlines(),
        lineterm='', fromfile='previous', tofile='current'
    )
    return '\n'.join(diff)


def get_kv(source_id, key, default=None):
    """Get a value from the per-source key-value store."""
    row = conn.execute(
        "SELECT value FROM kv WHERE source_id = ? AND key = ?",
        (source_id, key)
    ).fetchone()
    return json.loads(row[0]) if row else default


def set_kv(source_id, key, value):
    """Set a value in the per-source key-value store."""
    conn.execute(
        "INSERT OR REPLACE INTO kv (source_id, key, value) VALUES (?, ?, ?)",
        (source_id, key, json.dumps(value))
    )
    conn.commit()


# Quick test of helpers
test_hash = compute_hash("hello world")
print(f"SHA-256 of 'hello world': {test_hash[:24]}...")

test_diff = compute_diff("line 1\nline 2", "line 1\nline 2 modified\nline 3")
print(f"\nSample diff output:")
print(test_diff)

## Severity Classifier

We provide two classification strategies:

1. **Heuristic classifier**: Fast, no API calls, uses keyword matching. Good for initial filtering.
2. **LLM classifier**: Uses GPT-4o to understand context and provide nuanced severity ratings. Better for production use where accuracy matters.

The heuristic classifier is always used as a fallback. The LLM classifier is optional and can be enabled per tick.

In [ ]:
HIGH_SEVERITY_KEYWORDS = [
    'deprecated', 'removed', 'breaking', 'migration',
    'pricing', 'rate limit', 'security'
]
MEDIUM_SEVERITY_KEYWORDS = [
    'new', 'improved', 'preview', 'beta', 'released', 'added'
]


def classify_severity_heuristic(diff_text):
    """Classify change severity using keyword matching.
    
    Returns HIGH if any high-severity keywords are found,
    MEDIUM if any medium-severity keywords are found,
    LOW otherwise.
    """
    diff_lower = diff_text.lower()
    for kw in HIGH_SEVERITY_KEYWORDS:
        if kw in diff_lower:
            return "HIGH"
    for kw in MEDIUM_SEVERITY_KEYWORDS:
        if kw in diff_lower:
            return "MEDIUM"
    return "LOW"


def classify_severity_llm(diff_text, source_name):
    """Classify change severity using GPT-4o with Pydantic structured output.
    
    Returns a SeverityClassification with severity and reason.
    """
    response = client.responses.create(
        model=MODEL,
        instructions="You are a documentation change classifier.",
        input=f"""Classify the severity of this documentation change for developers.

Source: {source_name}
Changes:
{diff_text}

HIGH = breaking changes, deprecations, pricing changes, security issues
MEDIUM = new features, improvements, new models
LOW = formatting, minor updates, typo fixes""",
        text={
            "format": {
                "type": "json_schema",
                "name": "severity_classification",
                "strict": True,
                "schema": SeverityClassification.model_json_schema()
            }
        }
    )
    return SeverityClassification.model_validate_json(response.output_text)


# Test the heuristic classifier
print("Heuristic classifier tests:")
print(f"  'deprecated gpt-4' -> {classify_severity_heuristic('deprecated gpt-4')}")
print(f"  'new model released' -> {classify_severity_heuristic('new model released')}")
print(f"  'fixed typo in docs' -> {classify_severity_heuristic('fixed typo in docs')}")

## Approach 1: Stateful Worker (Pull-Based)

The **stateful worker** pattern is the core of this notebook. Each call to `agent_tick()` represents one cycle of the monitoring loop:

1. Iterate through all active sources
2. Check backoff status (skip sources with too many consecutive failures)
3. Fetch current content and compute its hash
4. Compare against the last known snapshot
5. If changed: compute diff, classify severity, store event, alert
6. If unchanged: log and skip

The function is **idempotent**: running it twice with no underlying changes produces no new events. This is critical for production agents that may be triggered by unreliable schedulers.

In [ ]:
def agent_tick(use_llm_classification=False):
    """
    One tick of the docs watcher agent.
    Fetches all active sources, checks for changes, creates events.
    
    This function is idempotent: running it twice with no changes
    produces no new events.
    
    Args:
        use_llm_classification: If True, use GPT-4o for severity classification.
                                If False, use fast heuristic keyword matching.
    """
    now = datetime.now().isoformat()
    sources = conn.execute("SELECT * FROM sources WHERE active = 1").fetchall()
    
    print(f"\n{'='*60}")
    print(f"Agent Tick at {now}")
    print(f"{'='*60}")
    
    for source in sources:
        source = dict(source)
        source_id = source['id']
        source_name = source['name']
        
        # Check backoff
        failure_count = get_kv(source_id, 'failure_count', 0)
        if failure_count >= 3:
            print(f"\n[{source_name}] Skipped - in backoff (failures: {failure_count})")
            continue
        
        print(f"\n[{source_name}]")
        
        try:
            # Fetch content
            content = fetch_content(source_id)
            content_hash = compute_hash(content)
            
            # Get last snapshot
            last = get_last_snapshot(source_id)
            
            if last and last['content_hash'] == content_hash:
                print(f"  No change (hash: {content_hash[:12]}...)")
                continue
            
            # Store new snapshot
            conn.execute(
                "INSERT INTO snapshots (source_id, fetched_at, content_hash, content_text) VALUES (?, ?, ?, ?)",
                (source_id, now, content_hash, content)
            )
            
            if last is None:
                print(f"  Initial snapshot stored (hash: {content_hash[:12]}...)")
                set_kv(source_id, 'last_alerted_hash', content_hash)
                conn.commit()
                continue
            
            # Change detected!
            diff_text = compute_diff(last['content_text'], content)
            
            # Check idempotency: don't alert if already alerted for this hash
            last_alerted = get_kv(source_id, 'last_alerted_hash')
            if last_alerted == content_hash:
                print(f"  Change detected but already alerted (idempotent skip)")
                continue
            
            # Classify severity
            if use_llm_classification:
                classification = classify_severity_llm(diff_text, source_name)
                severity = classification.severity
                summary = classification.reason
            else:
                severity = classify_severity_heuristic(diff_text)
                summary = f"Content changed ({severity} severity)"
            
            # Store event
            conn.execute(
                "INSERT INTO events (source_id, created_at, severity, summary, diff_text) VALUES (?, ?, ?, ?, ?)",
                (source_id, now, severity, summary, diff_text)
            )
            set_kv(source_id, 'last_alerted_hash', content_hash)
            set_kv(source_id, 'failure_count', 0)
            conn.commit()
            
            # Alert
            print(f"  CHANGE DETECTED! Severity: {severity}")
            print(f"  Summary: {summary}")
            print(f"  Diff preview: {diff_text[:200]}...")
            
        except Exception as e:
            failure_count += 1
            set_kv(source_id, 'failure_count', failure_count)
            conn.commit()
            print(f"  ERROR: {str(e)} (failure #{failure_count})")
    
    print(f"\n{'='*60}\n")

### Tick 1: Initial Snapshot

The first tick establishes the baseline. Since there are no previous snapshots, the agent stores the initial content for each source without generating any alerts. This is the "seed" step.

In [ ]:
agent_tick()  # Should store initial snapshots, no alerts

### Tick 2: No Changes

Running the agent again immediately should show "No change" for all sources, since the simulated content has not been updated. The hash comparison short-circuits any further processing.

In [ ]:
agent_tick()  # Should show "no change" for all sources

### Simulate Content Changes

Now we simulate what happens when monitored pages actually change. We update the OpenAI changelog (adding a breaking deprecation) and the Anthropic models page (adding a new model). The test source remains unchanged.

In [ ]:
# Simulate page updates
current_versions[1] = "v2"  # OpenAI changelog updated
current_versions[2] = "v2"  # Anthropic models page updated
print("Simulated content changes:")
print("  - OpenAI API Changelog: v1 -> v2 (breaking deprecation + new pricing)")
print("  - Anthropic Release Notes: v1 -> v2 (new model added)")
print("  - Local Test Source: unchanged (still v1)")

### Tick 3: Detect Changes with LLM Classification

This tick should detect the changes in sources 1 and 2, compute diffs, classify severity using GPT-4o, and store events. Source 3 should show no change.

In [ ]:
agent_tick(use_llm_classification=True)  # Should detect and classify changes

### Tick 4: Idempotency Check

Running the agent again without any new changes should produce NO new alerts. Even though the content differs from the original v1, the agent has already alerted on the current hash. This demonstrates idempotency -- the core property that makes the agent safe to run on unreliable schedules.

In [ ]:
agent_tick()  # Should NOT re-alert (same hashes, already alerted)

### Query All Detected Events

Let's inspect the events table to see all changes the agent has detected and classified.

In [ ]:
print("All detected events:")
print("-" * 60)
events = conn.execute("""
    SELECT e.*, s.name as source_name 
    FROM events e JOIN sources s ON e.source_id = s.id 
    ORDER BY e.created_at DESC
""").fetchall()

if not events:
    print("No events recorded yet.")
else:
    for event in events:
        event = dict(event)
        print(f"[{event['severity']}] {event['source_name']}: {event['summary']}")
        print(f"  Created: {event['created_at']}")
        print()

## Approach 2: Push-Based Mechanism

The stateful worker (Approach 1) uses a **pull-based** model: it polls sources on a schedule whether or not anything has changed. This is simple and works for any external page, but it wastes resources when nothing changes and has latency equal to the polling interval.

The **push-based** approach flips this: instead of polling, the agent exposes an endpoint (webhook) that receives notifications when content changes. This gives:

- **Near real-time** detection (no polling delay)
- **Zero wasted work** (only processes actual changes)
- **Lower resource usage** (no unnecessary fetches)

The trade-off is that the monitored system must support webhooks or change notifications. Many modern platforms (GitHub, Stripe, Slack) do, but arbitrary web pages do not.

Below we implement a `WebhookHandler` class that simulates this pattern. In production, this would be an HTTP server (e.g., Flask or FastAPI) that receives POST requests.

In [ ]:
class WebhookHandler:
    """
    Simulates a push-based webhook handler.
    In production, this would be an HTTP endpoint that receives
    POST requests when a monitored page changes.
    """
    def __init__(self, db_conn, openai_client):
        self.conn = db_conn
        self.client = openai_client
    
    def handle_webhook(self, payload):
        """
        Process an incoming webhook notification.
        
        Args:
            payload: dict with keys:
                - source_id (int): Which source changed
                - new_content (str): The updated page content
                - timestamp (str, optional): When the change occurred
        
        Returns:
            dict with status and details about what happened.
        """
        source_id = payload['source_id']
        new_content = payload['new_content']
        timestamp = payload.get('timestamp', datetime.now().isoformat())
        
        source = dict(self.conn.execute(
            "SELECT * FROM sources WHERE id = ?", (source_id,)
        ).fetchone())
        
        new_hash = compute_hash(new_content)
        last = get_last_snapshot(source_id)
        
        # Idempotency check
        if last and last['content_hash'] == new_hash:
            return {"status": "no_change", "message": "Content unchanged"}
        
        # Store snapshot
        self.conn.execute(
            "INSERT INTO snapshots (source_id, fetched_at, content_hash, content_text) VALUES (?, ?, ?, ?)",
            (source_id, timestamp, new_hash, new_content)
        )
        
        if last is None:
            self.conn.commit()
            return {"status": "initial", "message": "Initial snapshot stored"}
        
        # Process change
        diff_text = compute_diff(last['content_text'], new_content)
        classification = classify_severity_llm(diff_text, source['name'])
        
        self.conn.execute(
            "INSERT INTO events (source_id, created_at, severity, summary, diff_text) VALUES (?, ?, ?, ?, ?)",
            (source_id, timestamp, classification.severity, classification.reason, diff_text)
        )
        self.conn.commit()
        
        return {
            "status": "change_detected",
            "severity": classification.severity,
            "summary": classification.reason
        }


handler = WebhookHandler(conn, client)
print("WebhookHandler initialized and ready to receive events.")

### Test the Push-Based Handler

We simulate receiving a webhook notification for the test source with new content that includes a breaking change keyword. The handler should detect the change, classify its severity, and store an event.

In [ ]:
# Simulate receiving a webhook for the test source
result = handler.handle_webhook({
    "source_id": 3,
    "new_content": "Test content version 3 - push-based update with BREAKING changes",
    "timestamp": datetime.now().isoformat()
})
print(f"Webhook result: {json.dumps(result, indent=2)}")

In [ ]:
# Verify the event was stored
print("Updated events list (including push-based event):")
print("-" * 60)
events = conn.execute("""
    SELECT e.*, s.name as source_name 
    FROM events e JOIN sources s ON e.source_id = s.id 
    ORDER BY e.created_at DESC
""").fetchall()

for event in events:
    event = dict(event)
    print(f"[{event['severity']}] {event['source_name']}: {event['summary']}")
    print(f"  Created: {event['created_at']}")
    print()

## Comparing the Two Approaches

| Aspect | Stateful Worker (Pull) | Push-Based (Webhook) |
|--------|----------------------|---------------------|
| **Trigger** | Timer/schedule | External event |
| **Latency** | Depends on poll interval | Near real-time |
| **Complexity** | Simpler to implement | Needs webhook infrastructure |
| **Resource usage** | Polls even when nothing changes | Only processes on change |
| **Best for** | Monitoring external pages | Systems that support webhooks |
| **Idempotency** | Hash-based deduplication | Hash-based deduplication |
| **Error handling** | Backoff on repeated failures | Retry on webhook delivery failure |

In practice, many production systems use a **hybrid approach**: webhooks for systems that support them, and polling for external pages that do not. Both approaches share the same core logic (hash, diff, classify, store) -- only the trigger mechanism differs.

## Summary

In this notebook, we built a **Docs Watcher Agent** that demonstrates long-running agent loops with state persistence. Here are the key takeaways:

- **Stateful agents need persistent storage** (SQLite, Redis, PostgreSQL, etc.) to remember what they have seen across execution cycles.
- **Idempotency prevents duplicate alerts**: By tracking the last alerted hash per source, re-running the agent is always safe. Content hashing with SHA-256 makes change detection efficient.
- **Severity classification helps prioritize attention**: Combining fast heuristic rules with optional LLM classification gives both speed and accuracy.
- **The stateful worker pattern (poll, diff, classify, alert) is a universal monitoring primitive** that applies far beyond documentation watching.
- **Push-based mechanisms are more efficient when webhooks are available**, eliminating wasted polls and reducing detection latency to near real-time.
- **Backoff logic prevents hammering failed sources**, making the agent robust against transient and persistent failures.
- **This pattern generalizes to many domains**: security monitoring, competitor tracking, compliance checking, API changelog monitoring, price tracking, and more.